# 01 — Data Exploration

Khám phá hai dataset chính của Q-MMF:
- **Task 1 (MSA)**: MVSA-Single / CMU-MOSI — text + image + sentiment label
- **Task 2 (Captioning)**: Flickr8k — image + 5 captions/image

Mục tiêu:
1. Thống kê cơ bản (số samples, phân bố lớp)
2. Trực quan hóa samples (text + image)
3. Phân tích độ dài text/caption, tần suất từ
4. Verify DataLoader shapes theo milestone Tuần 1

In [ ]:
%matplotlib inline
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
print("Project root:", PROJECT_ROOT)

## 1. Cấu hình đường dẫn dataset

Cấu trúc thư mục dữ liệu kỳ vọng:

```
data/
├── mvsa/
│   ├── train.json, val.json, test.json
│   ├── data.json
│   └── images/
└── flickr8k/
    ├── train_annotations.json, val_annotations.json, test_annotations.json
    ├── annotations.json
    └── images/
```

In [ ]:
MVSA_DIR = DATA_DIR / "mvsa"
FLICKR_DIR = DATA_DIR / "flickr8k"


def check_dataset(base_dir, split_files):
    status = {}
    for f in split_files:
        p = base_dir / f
        status[f] = p.exists()
    img_dir = base_dir / "images"
    status["images/"] = img_dir.exists() and any(img_dir.iterdir()) if img_dir.exists() else False
    return status


print("MVSA     :", check_dataset(MVSA_DIR, ["data.json", "train.json", "val.json", "test.json"]))
print("Flickr8k :", check_dataset(FLICKR_DIR, ["annotations.json", "train_annotations.json", "val_annotations.json", "test_annotations.json"]))

if not MVSA_DIR.exists() and not FLICKR_DIR.exists():
    print("\n[!] Chưa có dữ liệu. Xem src/data/download.py để tải dataset.")

## 2. Task 1 — Multimodal Sentiment Analysis (MVSA-Single)

In [ ]:
def load_split(data_dir, split):
    path = Path(data_dir) / f"{split}.json"
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)


mvsa_train = load_split(MVSA_DIR, "train")
mvsa_val = load_split(MVSA_DIR, "val")
mvsa_test = load_split(MVSA_DIR, "test")

print(f"MVSA splits -> train: {len(mvsa_train)}, val: {len(mvsa_val)}, test: {len(mvsa_test)}")
if mvsa_train:
    print("Sample keys :", list(mvsa_train[0].keys()))
    print("Example     :", {k: str(v)[:80] for k, v in mvsa_train[0].items()})

In [ ]:
if mvsa_train:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, data) in zip(axes, [("Train", mvsa_train), ("Val", mvsa_val), ("Test", mvsa_test)]):
        labels = Counter(s.get("label", "?") for s in data)
        order = ["positive", "negative", "neutral"]
        counts = [labels.get(l, 0) for l in order]
        ax.bar(order, counts, color=["#4CAF50", "#F44336", "#9E9E9E"])
        ax.set_title(f"{name} ({len(data)})")
        ax.set_ylabel("count")
    plt.suptitle("MVSA-Single: Class distribution per split")
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có dữ liệu MVSA.")

In [ ]:
if mvsa_train:
    lengths = [len(s.get("text", "").split()) for s in mvsa_train]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(lengths, bins=40, color="#2196F3", edgecolor="white")
    axes[0].set_title("Text length distribution (words)")
    axes[0].set_xlabel("tokens")

    # Tokenized length vs max_length=128 cutoff
    axes[1].hist(np.clip(lengths, 0, 128), bins=40, color="#FF9800", edgecolor="white")
    axes[1].axvline(128, color="red", ls="--", label="max_length=128")
    axes[1].set_title("Clipped at BERT max_length")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    print(f"mean={np.mean(lengths):.1f}, median={np.median(lengths):.0f}, "
          f"p95={np.percentile(lengths, 95):.0f}, >128 words: {sum(l > 128 for l in lengths)}")

In [ ]:
# Hiển thị lưới ảnh mẫu kèm text + label
if mvsa_train:
    n_show = 6
    idxs = np.random.choice(len(mvsa_train), size=min(n_show, len(mvsa_train)), replace=False)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, i in zip(axes.flat, idxs):
        s = mvsa_train[i]
        img_path = MVSA_DIR / "images" / s.get("image", "")
        if img_path.exists():
            ax.imshow(Image.open(img_path).convert("RGB"))
        else:
            ax.set_facecolor("#eee")
        ax.set_title(f"[{s.get('label', '?')}] {str(s.get('text', ''))[:60]}...", fontsize=9)
        ax.axis("off")
    plt.suptitle("MVSA-Single: Random samples")
    plt.tight_layout()
    plt.show()

## 3. Task 2 — Image Captioning (Flickr8k)

In [ ]:
def load_flickr(data_dir, split="train"):
    path = Path(data_dir) / f"{split}_annotations.json"
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)


flickr_train = load_flickr(FLICKR_DIR, "train")
flickr_val = load_flickr(FLICKR_DIR, "val")
flickr_test = load_flickr(FLICKR_DIR, "test")

print(f"Flickr8k splits -> train: {len(flickr_train)}, val: {len(flickr_val)}, test: {len(flickr_test)}")
if flickr_train:
    n_imgs = len({s["image_id"] for s in flickr_train})
    print(f"Unique images in train: {n_imgs} (~{len(flickr_train) / n_imgs:.1f} captions/image)")
    print("Example:", {k: str(v)[:70] for k, v in flickr_train[0].items()})

In [ ]:
if flickr_train:
    cap_lens = [len(s["caption"].split()) for s in flickr_train]
    plt.figure(figsize=(10, 4))
    plt.hist(cap_lens, bins=40, color="#2196F3", edgecolor="white")
    plt.axvline(50, color="red", ls="--", label="max_caption_length=50")
    plt.title("Caption length distribution")
    plt.xlabel("words")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print(f"mean={np.mean(cap_lens):.1f}, p95={np.percentile(cap_lens, 95):.0f}")

    # Word frequency
    freq = Counter(w.lower().strip('.,!?') for s in flickr_train for w in s["caption"].split())
    top20 = freq.most_common(20)[::-1]
    plt.figure(figsize=(9, 6))
    plt.barh([w for w, _ in top20], [c for _, c in top20], color="#4CAF50")
    plt.title("Top-20 frequent words")
    plt.xlabel("count")
    plt.tight_layout()
    plt.show()

In [ ]:
# Xây dựng vocab bằng hàm trong src.data.preprocess (E-vocab cho captioning)
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocess import build_caption_vocab

if flickr_train:
    captions = [s["caption"] for s in flickr_train]
    vocab = build_caption_vocab(captions, min_freq=5)
    sizes = {}
    for mf in [1, 3, 5, 10]:
        sizes[mf] = len(build_caption_vocab(captions, min_freq=mf))
    print("Vocab size by min_freq:", sizes)

## 4. Verify DataLoader shapes (milestone Tuần 1)

Kỳ vọng:
- MSA: `input_ids [B,128]`, `attention_mask [B,128]`, `image [B,3,224,224]`, `label [B]`
- Caption: `image [B,3,224,224]`, `decoder_input_ids [B,50]`, `labels [B,50]`

In [ ]:
import torch
from torch.utils.data import DataLoader

try:
    from src.data.msa_dataset import MultimodalSentimentDataset
    from src.data.caption_dataset import ImageCaptionDataset

    if mvsa_train:
        ds = MultimodalSentimentDataset(str(MVSA_DIR), split="train", dataset_name="mvsa")
        dl = DataLoader(ds, batch_size=4, shuffle=False)
        batch = next(iter(dl))
        for k, v in batch.items():
            print(f"MSA    {k:>16}: {tuple(v.shape)}")

    if flickr_train:
        ds_cap = ImageCaptionDataset(str(FLICKR_DIR), split="train")
        dl_cap = DataLoader(ds_cap, batch_size=4, shuffle=False)
        bcap = next(iter(dl_cap))
        for k, v in bcap.items():
            print(f"Caption {k:>16}: {tuple(v.shape)}")
except FileNotFoundError as e:
    print("Bỏ qua verify (chưa có dữ liệu):", e)